In [1]:
include("../Envs/Env.jl")
include("../Algorithms/PPO-RNN.jl")

evaluate (generic function with 1 method)

#### Make sure Roomba is installed via
using Pkg

Pkg.add(url="https://github.com/sisl/RoombaPOMDPs.git")


## 1. Prepare Environment

In [2]:
using RoombaPOMDPs

max_speed = 5.0
speed_interval = 2.0
max_turn_rate = 1.0
turn_rate_interval = 0.2
action_space = vec([RoombaAct(v, om) for v in 0:speed_interval:max_speed, om in -max_turn_rate:turn_rate_interval:max_turn_rate])
pomdp = RoombaPOMDP(sensor=Lidar(), mdp=RoombaMDP(config=3, aspace=action_space, v_max=max_speed))
pomdp_name = "RoombaLidar"
bool_full_observability = false
env = Env(pomdp, bool_full_observability)
action_space = GetActionSpace(env)
function create_env()
    return Env(pomdp, bool_full_observability)
end

# # define convert_o function
# # function POMDPs.convert_o(T::Type{<:AbstractArray}, o::Int64, m::LightDark1D)
# #     vec = zeros(Float32, 3)
# #     vec[o] = 1.0f0
# #     return vec
# # end

# define process action function
function process_action(action_index::Int, action_space)
   return Float32.(action_space[action_index])
end


process_action (generic function with 1 method)

In [3]:
reset!(env)

1-element Vector{Float32}:
 0.0

In [4]:
actions(pomdp)

33-element Vector{RoombaAct}:
 [0.0, -1.0]
 [2.0, -1.0]
 [4.0, -1.0]
 [0.0, -0.8]
 [2.0, -0.8]
 [4.0, -0.8]
 [0.0, -0.6]
 [2.0, -0.6]
 [4.0, -0.6]
 [0.0, -0.4]
 [2.0, -0.4]
 [4.0, -0.4]
 [0.0, -0.2]
 ⋮
 [0.0, 0.4]
 [2.0, 0.4]
 [4.0, 0.4]
 [0.0, 0.6]
 [2.0, 0.6]
 [4.0, 0.6]
 [0.0, 0.8]
 [2.0, 0.8]
 [4.0, 0.8]
 [0.0, 1.0]
 [2.0, 1.0]
 [4.0, 1.0]

## 2. Prepare Parameters

In [5]:
state_dim = GetObsDim(env)
action_dim = 2
layer_size = 64
rnn_hidden_size = 64
gamma = discount(pomdp)
training_episodes = 10000
batch_size = 1024

[-18.101097480236703, -19.25349179432668, -1.7062436299374937, 0.0]
Float32[0.7008089]


1024

## 3. Prepare PPO-RNN agent

In [6]:
# if want to use gpu, need to uncomment the below line, and use device=Flux.gpu
# using CUDA

agent = PPORNNAgent(action_space, action_dim, state_dim;
    hidden_dim=layer_size, 
    rnn_hidden_size=rnn_hidden_size, 
    batch_size=batch_size, 
    device=Flux.cpu) 

PPORNNAgent(Chain(LSTM(3 => 64), Dense(64 => 64, tanh), Dense(64 => 33)), Chain(LSTM(3 => 64), Dense(64 => 64, tanh), Dense(64 => 1)), (layers = ((cell = (Wi = Leaf(Adam(eta=0.0001, beta=(0.9, 0.999), epsilon=1.0e-8), (Float32[0.0 0.0 0.0; 0.0 0.0 0.0; … ; 0.0 0.0 0.0; 0.0 0.0 0.0], Float32[0.0 0.0 0.0; 0.0 0.0 0.0; … ; 0.0 0.0 0.0; 0.0 0.0 0.0], (0.9, 0.999))), Wh = Leaf(Adam(eta=0.0001, beta=(0.9, 0.999), epsilon=1.0e-8), (Float32[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], Float32[0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0], (0.9, 0.999))), bias = Leaf(Adam(eta=0.0001, beta=(0.9, 0.999), epsilon=1.0e-8), (Float32[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], Float32[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0  …  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], (0.9, 0.999)))),), (weight = Leaf(Adam(eta=0.0001, beta=(0.9, 0.999), epsilon=1.0e-8), 

## 4. Train

In [ ]:
# 训练
rewards, losses, evals = train!(create_env, agent, training_episodes)

Progress:   0%|█                                        |  ETA: 19:08:3233m

## 5. Evaluation

In [ ]:
evaluate(env, agent; num_episodes=10000, max_steps=100) 

## (Todo) Save or plot the data from Train (rewards, losses, evals)